In [ ]:
!pip install transformers datasets streamlit --quiet

In [ ]:
import os
import torch
import zipfile
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    pipeline
)

In [ ]:
mlqa_configs = [
    "mlqa.en.ar", "mlqa.en.de", "mlqa.en.es",
    "mlqa.en.hi", "mlqa.en.vi", "mlqa.en.zh"
]
mlqa_datasets = {config.split('.')[-1]: load_dataset("mlqa", config) for config in mlqa_configs}

# Combine all test splits into one dataset for training
combined_dataset = concatenate_datasets([ds["test"] for ds in mlqa_datasets.values()])
combined_dataset = combined_dataset.shuffle(seed=42)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
model_checkpoint = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
def prepare_features(examples):
    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=384,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")

    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        sequence_ids = tokenized_examples.sequence_ids(i)
        sample_index = sample_mapping[i]

        answers = examples["answers"][sample_index]
        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1
        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
        else:
            for idx in range(token_start_index, token_end_index + 1):
                if offsets[idx][0] <= start_char < offsets[idx][1]:
                    start_position = idx
                if offsets[idx][0] < end_char <= offsets[idx][1]:
                    end_position = idx
                    break

            tokenized_examples["start_positions"].append(start_position)
            tokenized_examples["end_positions"].append(end_position)

    return tokenized_examples

tokenized_dataset = combined_dataset.map(
    prepare_features,
    batched=True,
    remove_columns=combined_dataset.column_names
)

In [ ]:
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

training_args = TrainingArguments(
    output_dir="./qa_model",
    evaluation_strategy="no",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    num_train_epochs=20,
    weight_decay=0.01,
    save_total_limit=1,
    save_strategy="epoch",
    logging_dir="./logs",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

Some weights of XLMRobertaForQuestionAnswering were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# --------------------------------------------------------
# ✅ 7. Train the model
# --------------------------------------------------------
trainer.train()

# --------------------------------------------------------
# ✅ 8. Save model and tokenizer
# --------------------------------------------------------
output_dir = "multilingual-qa-model"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

# --------------------------------------------------------
# ✅ 9. Zip the model folder for upload to Hugging Face
# --------------------------------------------------------
def zip_files(directory_to_zip, zip_filename):
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(directory_to_zip):
            for file in files:
                zipf.write(os.path.join(root, file),
                           os.path.relpath(os.path.join(root, file),
                                           os.path.join(directory_to_zip, '..')))

zip_files(output_dir, "multilingual-qa-model.zip")

print("✅ Model saved and zipped as multilingual-qa-model.zip")

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: anirudhjeevan1999 (anirudh2857) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,3.716000
1000,2.576700
1500,1.855700
2000,1.630000
2500,1.520300
3000,1.404200
3500,1.328900
4000,1.282300
4500,1.194400
5000,1.159100


✅ Model saved and zipped as multilingual-qa-model.zip
